# 02 — Graph Construction

Builds the knowledge graph from CSV data using NetworkX and visualises it. Also includes the commands to push the graph to a Neo4j instance if one is running.

In [ ]:
import sys; sys.path.insert(0, '..')
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src.graph.builder import load_data, build_networkx_graph

workers, platforms, skills, districts, edges = load_data(use_synthetic=True)
G = build_networkx_graph(workers, platforms, skills, districts, edges)

print('Nodes:', G.number_of_nodes())
print('Edges:', G.number_of_edges())
print()
print('Node types:', set(nx.get_node_attributes(G, 'node_type').values()))

In [ ]:
# Basic graph statistics
print('Degree centrality (top 5):')
deg = nx.degree_centrality(G)
for node, score in sorted(deg.items(), key=lambda x: -x[1])[:5]:
    ntype = G.nodes[node].get('node_type', '')
    label = G.nodes[node].get('label', G.nodes[node].get('name', node))
    print(f'  {label} ({ntype}): {score:.3f}')

In [ ]:
# Visualise the graph
pos = nx.spring_layout(G, seed=42, k=2.5)

type_to_color = {
    'worker':   'steelblue',
    'platform': 'tomato',
    'skill':    'mediumpurple',
    'district': 'mediumseagreen',
}

node_colors = [type_to_color.get(G.nodes[n].get('node_type', ''), 'grey') for n in G.nodes()]
node_labels = {
    n: G.nodes[n].get('label', G.nodes[n].get('name', n))[:15]
    for n in G.nodes()
}

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx(
    G, pos=pos, ax=ax, labels=node_labels,
    node_color=node_colors, node_size=1000,
    font_size=7, edge_color='#cccccc',
    arrows=True, arrowsize=12,
)

legend_handles = [mpatches.Patch(color=c, label=t) for t, c in type_to_color.items()]
ax.legend(handles=legend_handles, loc='upper left')
ax.set_title('Ghost Worker Knowledge Graph')
ax.axis('off')
plt.tight_layout()
plt.savefig('../logs/graph_viz.png', bbox_inches='tight')
plt.show()

In [ ]:
# To push to Neo4j, uncomment the line below.
# Neo4j must be running locally with credentials set in src/utils/config.py.
#
# from src.graph.builder import push_to_neo4j
# push_to_neo4j(workers, platforms, skills, districts, edges)

print('Graph construction complete. Push to Neo4j when instance is ready.')